In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,
recall_score,f1_score,roc_auc_score,precision_score)

In [58]:
df=pd.read_csv(r'C:\Users\adnan\OneDrive\Desktop\5TH SEM OJT\project 1_diabaties\data\processed\mohammed_adnan_p1w2_features.csv')

In [59]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi_band','glucose_band']
x = df[feature_cols].copy()
y = df["outcome"]
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi_band,glucose_band
0,79,0,124.000000,71.0,30.000000,134.67525,23.4,0.588,Normal,Prediabetes
1,37,0,153.000000,85.0,24.000000,42.00000,33.9,0.192,Obese,Diabetes
2,39,0,142.000000,68.0,22.000000,159.00000,26.9,0.777,Overweight,Diabetes
3,68,7,121.000000,88.0,26.406623,134.67525,29.0,1.217,Overweight,Prediabetes
4,75,0,107.000000,84.0,31.000000,101.00000,21.2,0.278,Normal,Prediabetes
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.855165,81.0,34.000000,134.67525,40.3,0.641,Obese,Prediabetes
946,71,0,140.000000,64.0,37.000000,134.67525,33.4,0.116,Obese,Diabetes
947,40,0,130.000000,73.0,26.406623,258.00000,20.1,0.709,Normal,Diabetes
948,52,4,123.000000,76.0,22.000000,189.00000,38.9,0.540,Obese,Prediabetes


In [60]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [61]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [62]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [63]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [64]:
def score_row(name, pred, score=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found": int(tp),
        "missed": int(fn),
        "false_alarms": int(fp)
    }
    if score is not None:
        r["auc"] = round(roc_auc_score(y_test, score), 4)
    return r

print("Train:", x_train.shape ,"Test:", x_test.shape)
print("DIABETIC PATIENTS IN THE TEST SET:", int(np.sum(y_test)), "of", len(y_test))
for nm, m, xt in [("Logistic Regression", logreg, x_test_scaled),
                  ("Decision Tree", dtree, x_test),
                  ("KNN", knn, x_test_scaled)]:
    print(f"{nm:22s}  accuracy {accuracy_score(y_test,m.predict(xt)):.4f}")

Train: (760, 13) Test: (190, 13)
DIABETIC PATIENTS IN THE TEST SET: 55 of 190
Logistic Regression     accuracy 0.7579
Decision Tree           accuracy 0.7211
KNN                     accuracy 0.7368


In [65]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182

print("PROJECT METRIC : recall  (guard: F1>=", GUARD_F1,")")
print("TARGET : recall >=", TARGET_RECALL)
print("BEST_SO_FAR :", BEST_SO_FAR, "(WEEK 5 BASELINE AT THRESHOLD 0.30)")
print("GAP TO CLOSE :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall  (guard: F1>= 0.5 )
TARGET : recall >= 0.7
BEST_SO_FAR : 0.6182 (WEEK 5 BASELINE AT THRESHOLD 0.30)
GAP TO CLOSE : 0.0818


Explain how an SVM reaches a dcision


SVM finds a line that separates two groups.

It finds the best separating line.
The closest points to the line are called support vectors.
For a new data point, SVM checks which side of the line it is on.
Then it assigns the point to that class

In [66]:
from sklearn.svm import SVC


In [67]:
svm = SVC(random_state=42)
svm.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("fitted:", svm_rbf)
print("patients flagged:",int(pred_rbf.sum()), "of", len(pred_rbf))

fitted: SVC(random_state=42)
patients flagged: 1 of 190


In [68]:
row_rbf = score_row('SVC rbf ', pred_rbf,svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print('Target recall:', TARGET_RECALL,'| Best so far:', BEST_SO_FAR)
print('This model :', row_rbf['recall'])

   model  accuracy  precision  recall  f1  found  missed  false_alarms    auc
SVC rbf     0.7053        0.0     0.0 0.0      0      55             1 0.5296

Target recall: 0.7 | Best so far: 0.6182
This model : 0.0
